# Lite3 detector: auto-label video frames, train, export `detector.onnx`

**Runtime → Change runtime type → T4 GPU** (training on CPU would take hours). Then run the cells top to bottom.

Your zips, labels, training checkpoints and `detector.onnx` are saved to **Google Drive → MyDrive/lite3**.
If Colab disconnects, just run all cells again from the top: it re-uses what is on Drive and **resumes training** where it stopped.

1. Upload `lite3_frames.zip`
2. Auto-label with a big YOLO-World model (people only count as `survivor` if they are inside a printed poster)
3. Check the preview (and optionally fix labels in Roboflow)
4. Train a small fast YOLOv8 on the labels
5. Export `detector.onnx` and download it

In [ ]:
!pip install -q ultralytics onnx onnxslim albumentations
import os, glob, random, shutil, zipfile, collections
import cv2, numpy as np, torch, torchvision
from ultralytics import YOLO
print("GPU:", torch.cuda.is_available(), "  <- if False: Runtime > Change runtime type > T4 GPU (CPU training takes hours)")

# Everything important is saved to Google Drive, so a disconnect does not lose your work.
from google.colab import drive, files
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/lite3"
os.makedirs(f"{WORK}/zips", exist_ok=True)
print("saving to", WORK)

## 1. Upload the frames zip

In [ ]:
saved = sorted(glob.glob(f"{WORK}/zips/*.zip"))
if saved:
    print("using zips already on Drive:", [os.path.basename(z) for z in saved])
    print("(to upload new ones instead, delete them from MyDrive/lite3/zips first)")
else:
    up = files.upload()                  # select ALL frame zips (e.g. lite3_frames.zip + robot_frames.zip)
    for zname in up:
        shutil.copy(zname, f"{WORK}/zips/")
    saved = sorted(glob.glob(f"{WORK}/zips/*.zip"))
shutil.rmtree("frames", ignore_errors=True)
for zname in saved:
    with zipfile.ZipFile(zname) as z: z.extractall(".")
imgs = sorted(glob.glob("frames/*.jpg"))
print(len(imgs), "frames")

## 2. Auto-label
Edit `CLASSES` to add/remove objects. Left side = class name the robot will report,
right side = words YOLO-World searches for. Add more words if an object is missed.

In [ ]:
# STANDARD LABELS (left) <- words YOLO-World searches for (right). Only these names are reported.
CLASSES = {
    "survivor":     ["person"],                      # kept ONLY if inside a poster (see HELPERS)
    "medical_kit":  [# the arena's medical station is a toy white-and-teal "MEDICAL TROLLY" cart
                     "medical trolley", "medical cart", "doctor cart", "hospital cart",
                     "toy doctor play set", "white and teal toy medical cart",
                     # tools on / around it, and classic first-aid items
                     "stethoscope", "toy syringe", "thermometer", "medical scissors",
                     "first aid kit", "medical kit", "red cross box", "medicine box",
                     "pill bottle", "medicine bottle", "bandage"],
    "hand_tools":   ["power drill", "toy drill", "wrench", "spanner", "hammer", "screwdriver",
                     "pliers", "hand saw", "screw", "bolt", "gear"],
    "food":         ["banana", "orange fruit", "grapes", "apple", "bell pepper", "carrot", "egg",
                     "bread", "tin can", "canned food", "food box"],
    "water":        ["water bottle", "bottle of water", "drink can", "juice box"],
    "utensils":     ["spoon", "fork", "kitchen knife", "cooking pot", "frying pan", "spatula",
                     "plate", "cup", "bowl"],
    # context only (help the LLM guess the room type); listed in ignore_classes on the robot
    "cart":         ["shopping cart", "shopping trolley"],
    "market_stall": ["toy market stall", "shop counter"],
    "tool_bench":   ["toy workbench", "tool bench"],
    # look-alikes of the medical trolley: give them their own class so they are NOT labelled medical
    "cleaning_cart": ["cleaning trolley", "janitor cart", "wet floor sign", "mop bucket"],
    "kitchen":      ["toy kitchen", "toy oven", "dishwasher", "kitchen cabinet"],
}
# a smaller box of the same class lying inside a bigger one is dropped
# (e.g. the stethoscope ON the medical trolley -> keep just the trolley box)
NESTED_INSIDE = 0.8
HELPERS = ["poster", "paper sheet", "printed photo"]   # only used to decide which people are pictures
CONF = collections.defaultdict(lambda: 0.20, {"hand_tools": 0.18, "survivor": 0.25, "medical_kit": 0.12,
                                               "market_stall": 0.30, "tool_bench": 0.30,
                                               "cleaning_cart": 0.25, "kitchen": 0.30})
HELPER_CONF = 0.15
VICTIM_INSIDE = 0.6          # fraction of the person box that must lie inside a poster box

names = list(CLASSES)
prompts, owner = [], []
for ci, c in enumerate(names):
    for p in CLASSES[c]:
        prompts.append(p); owner.append(ci)
for h in HELPERS:
    prompts.append(h); owner.append(-1)

if torch.cuda.is_available():
    wm, LABEL_IMGSZ = YOLO("yolov8x-worldv2.pt"), 1280   # best labels, ~0.3 s/frame on a T4
else:
    print("NO GPU: using a smaller labeller so this finishes in reasonable time (training still needs a GPU)")
    wm, LABEL_IMGSZ = YOLO("yolov8m-worldv2.pt"), 960
wm.set_classes(prompts)

def inside(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    return ix * iy / max(1e-6, (a[2] - a[0]) * (a[3] - a[1]))

REDO_LABELS = False          # set True after changing CLASSES / thresholds (wipes saved labels)
DRIVE_LABELS = f"{WORK}/autolabels"
if REDO_LABELS:
    shutil.rmtree(DRIVE_LABELS, ignore_errors=True)
os.makedirs(DRIVE_LABELS, exist_ok=True)
shutil.rmtree("autolabels", ignore_errors=True); shutil.copytree(DRIVE_LABELS, "autolabels")
lab = lambda f: os.path.splitext(os.path.basename(f))[0] + ".txt"
imgs_to_label = [f for f in imgs if not os.path.exists(f"autolabels/{lab(f)}")]   # resume: skip done frames
print(f"{len(imgs) - len(imgs_to_label)} frames already labelled (saved on Drive), {len(imgs_to_label)} to go")
from tqdm.auto import tqdm
stats = collections.Counter(); dropped_people = 0
for n_done, f in enumerate(tqdm(imgs_to_label, desc="auto-labelling")):
    r = wm.predict(f, imgsz=LABEL_IMGSZ, conf=0.10, iou=0.5, verbose=False)[0]
    h, w = r.orig_shape
    xyxy = r.boxes.xyxy.cpu(); conf = r.boxes.conf.cpu(); pid = r.boxes.cls.cpu().int().tolist()
    cls = torch.tensor([owner[i] for i in pid], dtype=torch.long)
    helpers = [xyxy[i].tolist() for i in range(len(pid)) if cls[i] == -1 and conf[i] >= HELPER_CONF]
    keep = []
    for i in range(len(pid)):
        c = int(cls[i])
        if c < 0 or conf[i] < CONF[names[c]]:
            continue
        if names[c] == "survivor" and not any(inside(xyxy[i].tolist(), hb) >= VICTIM_INSIDE for hb in helpers):
            dropped_people += 1          # a real person (bystander), not a picture
            continue
        keep.append(i)
    lines = []
    if keep:
        k = torch.tensor(keep)
        nms = torchvision.ops.batched_nms(xyxy[k], conf[k], cls[k], 0.5)
        kept_j = k[nms].tolist()
        area = lambda b: (b[2] - b[0]) * (b[3] - b[1])
        final = []
        for j in kept_j:
            bj = xyxy[j].tolist()
            if any(int(cls[o]) == int(cls[j]) and o != j and area(xyxy[o].tolist()) > area(bj)
                   and inside(bj, xyxy[o].tolist()) >= NESTED_INSIDE for o in kept_j):
                continue
            final.append(j)
        for j in final:
            x1, y1, x2, y2 = xyxy[j].tolist(); c = int(cls[j])
            lines.append(f"{c} {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}")
            stats[names[c]] += 1
    txt = "\n".join(lines)
    open(f"autolabels/{lab(f)}", "w").write(txt)
    open(f"{DRIVE_LABELS}/{lab(f)}", "w").write(txt)       # saved to Drive immediately -> safe to stop anytime
print("boxes per class (this run):", dict(stats))
print("real people ignored (not inside a poster):", dropped_people)
print("total labelled:", len(glob.glob("autolabels/*.txt")), "/", len(imgs))

## 3. Preview the labels (look for wrong / missing boxes)

In [ ]:
import matplotlib.pyplot as plt
def show(img_paths, label_dir, n=12, seed=0):
    random.seed(seed); sel = random.sample(img_paths, min(n, len(img_paths)))
    plt.figure(figsize=(20, 12))
    for k, f in enumerate(sel):
        im = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
        lf = f"{label_dir}/{os.path.splitext(os.path.basename(f))[0]}.txt"
        for ln in open(lf).read().split("\n") if os.path.exists(lf) else []:
            if not ln.strip(): continue
            c, cx, cy, bw, bh = ln.split(); cx, cy, bw, bh = map(float, (cx, cy, bw, bh))
            x1, y1, x2, y2 = int((cx-bw/2)*w), int((cy-bh/2)*h), int((cx+bw/2)*w), int((cy+bh/2)*h)
            col = (255, 0, 0) if names[int(c)] == "survivor" else (0, 255, 0)
            cv2.rectangle(im, (x1, y1), (x2, y2), col, 3)
            cv2.putText(im, names[int(c)], (x1, max(20, y1-6)), 0, 0.9, col, 2)
        plt.subplot(3, 4, k+1); plt.imshow(im); plt.axis("off"); plt.title(os.path.basename(f))
    plt.tight_layout(); plt.show()
show(imgs, "autolabels", seed=0)

In [ ]:
show(imgs, "autolabels", seed=1)   # run again with other seeds to look at more frames

### Optional but recommended: fix the labels in Roboflow (15-30 min)
Auto-labels miss some objects and add some wrong boxes, and the model learns those mistakes.
1. Run the next cell, download `autolabeled_dataset.zip`.
2. roboflow.com → new Object Detection project → upload the zip (it keeps the boxes) → fix boxes → Generate → Export **YOLOv8** → "show download code".
3. Paste that code into the cell after it and run it. Skip both cells to train on the raw auto-labels.

In [ ]:
shutil.rmtree("rf", ignore_errors=True); os.makedirs("rf/images"); os.makedirs("rf/labels")
for f in imgs:
    b = os.path.splitext(os.path.basename(f))[0]
    shutil.copy(f, f"rf/images/{b}.jpg"); shutil.copy(f"autolabels/{b}.txt", f"rf/labels/{b}.txt")
open("rf/data.yaml", "w").write(f"names: {names}\nnc: {len(names)}\n")
shutil.make_archive("autolabeled_dataset", "zip", "rf"); files.download("autolabeled_dataset.zip")

In [ ]:
# --- ONLY if you fixed labels in Roboflow: paste its download code here, e.g.
# !pip install -q roboflow
# from roboflow import Roboflow
# ds = Roboflow(api_key="...").workspace("...").project("...").version(1).download("yolov8")
# DATA_YAML = f"{ds.location}/data.yaml"

## 4. Build the train/val split and train

In [ ]:
if "DATA_YAML" not in globals():
    # split by time inside each video: last 20% of each video -> val (avoids near-duplicate leakage)
    shutil.rmtree("ds", ignore_errors=True)
    for s in ("train", "val"):
        os.makedirs(f"ds/images/{s}"); os.makedirs(f"ds/labels/{s}")
    byvid = collections.defaultdict(list)
    for f in imgs: byvid[os.path.basename(f).split("_")[0]].append(f)
    for v, fs in byvid.items():
        cut = int(len(fs) * 0.8)
        for i, f in enumerate(fs):
            s = "train" if i < cut else "val"; b = os.path.splitext(os.path.basename(f))[0]
            shutil.copy(f, f"ds/images/{s}/{b}.jpg"); shutil.copy(f"autolabels/{b}.txt", f"ds/labels/{s}/{b}.txt")
    DATA_YAML = os.path.abspath("ds/data.yaml")
    open(DATA_YAML, "w").write(f"path: {os.path.abspath('ds')}\ntrain: images/train\nval: images/val\nnames: {names}\n")
print(open(DATA_YAML).read())

RUN = f"{WORK}/runs/lite3"
last = f"{RUN}/weights/last.pt"
if os.path.exists(last) and not os.path.exists(f"{RUN}/weights/best_done.flag"):
    print("resuming interrupted training from", last)
    YOLO(last).train(resume=True)
else:
    model = YOLO("yolov8s.pt")   # "yolov8n.pt" = ~2x faster on the robot CPU, a bit less accurate
    model.train(data=DATA_YAML, epochs=120, patience=40, imgsz=640, batch=16,
                degrees=5, translate=0.1, scale=0.5, fliplr=0.5, close_mosaic=10,
                project=f"{WORK}/runs", name="lite3", exist_ok=True)
open(f"{RUN}/weights/best_done.flag", "w").write("done")
print("trained model saved on Drive:", f"{RUN}/weights/best.pt")

## 5. Check results

In [ ]:
best = YOLO(f"{WORK}/runs/lite3/weights/best.pt")
m = best.val(data=DATA_YAML, imgsz=640)
for i, n in best.names.items():
    print(f"{n:14s} mAP50 = {m.box.maps[i]:.2f}" if i < len(m.box.maps) else n)

In [ ]:
root = os.path.dirname(DATA_YAML)
val_imgs = sorted(glob.glob(f"{root}/valid/images/*.jpg") + glob.glob(f"{root}/images/val/*.jpg"))
res = best.predict(random.sample(val_imgs, min(8, len(val_imgs))), imgsz=640, conf=0.3, verbose=False)
plt.figure(figsize=(20, 10))
for k, r in enumerate(res):
    plt.subplot(2, 4, k+1); plt.imshow(r.plot()[:, :, ::-1]); plt.axis("off")
plt.tight_layout(); plt.show()

## 6. Export for the robot and download

In [ ]:
best = YOLO(f"{WORK}/runs/lite3/weights/best.pt")
onnx_path = best.export(format="onnx", imgsz=640, opset=12, simplify=True)
shutil.copy(onnx_path, "detector.onnx"); shutil.copy(onnx_path, f"{WORK}/detector.onnx")
import onnxruntime as ort
s = ort.InferenceSession("detector.onnx")
print(s.get_inputs()[0].shape, s.get_outputs()[0].shape)
print(s.get_modelmeta().custom_metadata_map.get("names"))
files.download("detector.onnx")